In [1]:
# Chunking stratagies
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

big_paragraph = (
    "AI automates repetitive tasks and saves time, working continuously without breaks while "
    "reducing human error in routine processes. It quickly detects patterns in large datasets, "
    "improves decision-making with data insights, and personalizes user experiences across apps. "
    "Organizations use AI to scale services to millions of users, speed up support with smart "
    "chatbots, lower operational costs through efficiency, and enable rapid prototyping and "
    "experimentation. In practice, success depends on the quality of data, clear objectives, and "
    "robust evaluation. Teams must also consider fairness, privacy, security, and transparency "
    "to build trust and maintain compliance. When integrated thoughtfully, AI augments human "
    "capabilities, unlocking new workflows, richer analytics, and faster iteration cycles that "
    "compound over time in both product and process improvements. As adoption grows, organizations "
    "refine governance, monitoring, and feedback loops, ensuring systems remain reliable, safe, "
    "and aligned with real user needs while continuously learning from fresh data and shifting "
    "market conditions."
) * 5 

big_paragraph

'AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes. It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps. Organizations use AI to scale services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation. In practice, success depends on the quality of data, clear objectives, and robust evaluation. Teams must also consider fairness, privacy, security, and transparency to build trust and maintain compliance. When integrated thoughtfully, AI augments human capabilities, unlocking new workflows, richer analytics, and faster iteration cycles that compound over time in both product and process improvements. As adoption grows, organizations refine governance, monitoring, and feedback loops, ensuring systems remain reliable, safe, and alig

In [ ]:
# Fixed length
# Section by section or paragraph 
# Some spliter based on (.)
# Context chucking

In [3]:
def chunk_fixed_no_overlap(text: str, size: int = 300):
    return [text[i:i+size] for i in range(0, len(text), size)]

In [4]:
chunk_fixed_no_overlap(big_paragraph)

['AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes. It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps. Organizations use AI to scale ',
 'services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation. In practice, success depends on the quality of data, clear objectives, and robust evaluation. Teams must also consider fairness, privacy,',
 ' security, and transparency to build trust and maintain compliance. When integrated thoughtfully, AI augments human capabilities, unlocking new workflows, richer analytics, and faster iteration cycles that compound over time in both product and process improvements. As adoption grows, organizations ',
 'refine governance, monitoring, and feedback loops, ensuring systems remain reliable

In [2]:
def chunk_slide(text, size=80, overlap_pct=0.10):
    words = text.split()
    n = len(words)
    i = 0
    chunks = []
    back = max(1, int(size * overlap_pct))  # e.g., 10% of size

    while i < n:
        end = min(i + size, n)
        chunks.append(" ".join(words[i:end]))
        if end == n:
            break
        i = end - back  # slide back by 10–15% of the last chunk length

    return chunks


In [3]:
len(chunk_slide(big_paragraph))

11

In [7]:
chunk_data = chunk_slide(big_paragraph)
chunk_data

['AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes. It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps. Organizations use AI to scale services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation. In practice, success depends on the quality of data, clear objectives, and robust evaluation. Teams must also',
 'clear objectives, and robust evaluation. Teams must also consider fairness, privacy, security, and transparency to build trust and maintain compliance. When integrated thoughtfully, AI augments human capabilities, unlocking new workflows, richer analytics, and faster iteration cycles that compound over time in both product and process improvements. As adoption grows, organizations refine governance, monitoring, and fe

In [5]:
chunk_slide(big_paragraph)[2]

'data and shifting market conditions.AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes. It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps. Organizations use AI to scale services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation. In practice, success depends on the quality of data, clear objectives, and robust'

In [10]:
def hierarchical_chunking(text):
    """
    Split document using structured section headers (e.g., "Section 1:", "Section 2:").
    """
    sections = re.split(r'\n(?=Section \d+:)', text.strip())
    return [sec.strip() for sec in sections if sec.strip()]

In [13]:
#!pip install nltk
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


from nltk.tokenize import sent_tokenize

def semantic_chunking(text, threshold=0.75):
    """
    Group nearby sentences based on semantic similarity (using cosine similarity).
    """
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = sent_tokenize(text)
    embeddings = model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Harsha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Harsha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [15]:
len(semantic_chunking(big_paragraph))

31

In [14]:
semantic_chunking(big_paragraph)

['AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes.',
 'It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps.',
 'Organizations use AI to scale services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation.',
 'In practice, success depends on the quality of data, clear objectives, and robust evaluation.',
 'Teams must also consider fairness, privacy, security, and transparency to build trust and maintain compliance.',
 'When integrated thoughtfully, AI augments human capabilities, unlocking new workflows, richer analytics, and faster iteration cycles that compound over time in both product and process improvements.',
 'As adoption grows, organizations refine governance, monitoring, and feedback loops, ensuring systems remain

In [16]:
# pip install langchain-text-splitters tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "AI automates repetitive tasks and saves time, working continuously without breaks while "
    "reducing human error in routine processes. It quickly detects patterns in large datasets, "
    "improves decision-making with data insights, and personalizes user experiences across apps. "
    "Organizations use AI to scale services to millions of users, speed up support with smart "
    "chatbots, lower operational costs through efficiency, and enable rapid prototyping and "
    "experimentation. In practice, success depends on the quality of data, clear objectives, and "
    "robust evaluation. Teams must also consider fairness, privacy, security, and transparency "
    "to build trust and maintain compliance. "
) * 5  # make it long

# 10% overlap of 300 chars → 30
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_text(text)

print("num_chunks:", len(chunks))
print("first_chunk_len:", len(chunks[0]))
print("first_chunk:\n", chunks[0])


num_chunks: 13
first_chunk_len: 299
first_chunk:
 AI automates repetitive tasks and saves time, working continuously without breaks while reducing human error in routine processes. It quickly detects patterns in large datasets, improves decision-making with data insights, and personalizes user experiences across apps. Organizations use AI to scale


In [17]:
chunks[1]

'Organizations use AI to scale services to millions of users, speed up support with smart chatbots, lower operational costs through efficiency, and enable rapid prototyping and experimentation. In practice, success depends on the quality of data, clear objectives, and robust evaluation. Teams must'